In [ ]:
pip install textblob


In [ ]:
import pandas as pd
from textblob import TextBlob
import plotly.express as px

# 讀取包含歌詞的 CSV 文件
df = pd.read_csv('data/stanza_annotations.csv')  # 替換為你的 CSV 文件名

# 對每段歌詞進行情感分析
polarity_scores = []
subjectivity_scores = []

for text in df['Stanza Text']:
    analysis = TextBlob(text)
    polarity_scores.append(analysis.sentiment.polarity)
    subjectivity_scores.append(analysis.sentiment.subjectivity)

# 添加情感分析結果到 DataFrame
df['Polarity'] = polarity_scores
df['Subjectivity'] = subjectivity_scores

# 顯示前幾行以驗證數據
print(df.head())

# 步驟 4: 可視化情感分析結果
# 使用 Plotly 繪製極性得分的柱狀圖
fig_polarity = px.bar(
    df,
    x='Stanza Number',
    y='Polarity',
    color='Polarity',
    title='Polarity of Stanzas',
    labels={'stanza number': 'Stanza Number', 'Polarity': 'Polarity Score'},
    color_continuous_scale='RdYlGn'  # 使用紅黃綠顏色來顯示情感的變化
)

fig_polarity.update_layout(
    xaxis=dict(title='Stanza Number'),
    yaxis=dict(title='Polarity (Sentiment)', range=[-1, 1]),
    width=800,
    height=500,
    title_font_size=20,
)

fig_polarity.show()

# 可選：可視化主觀性得分
fig_subjectivity = px.scatter(
    df,
    x='Stanza Number',
    y='Subjectivity',
    color='Subjectivity',
    title='Subjectivity of Stanzas',
    labels={'stanza number': 'Stanza Number', 'Subjectivity': 'Subjectivity Score'},
    color_continuous_scale='Blues'
)

fig_subjectivity.update_layout(
    xaxis=dict(title='Stanza Number'),
    yaxis=dict(title='Subjectivity (0 = Objective, 1 = Subjective)', range=[0, 1]),
    width=800,
    height=500,
    title_font_size=20,
)

fig_subjectivity.show()


In [ ]:
from textblob import TextBlob
import plotly.graph_objects as go
from plotly.subplots import make_subplots
# 對每段歌詞進行情感分析，計算主觀性
df['Subjectivity'] = df['Stanza Text'].apply(lambda text: TextBlob(text).sentiment.subjectivity)

# 獲取所有唯一的歌曲名稱
unique_songs = df['Song title'].unique()

# 設置每行顯示的子圖數量，例如 4
num_cols = 4
num_rows = len(unique_songs) // num_cols + (1 if len(unique_songs) % num_cols > 0 else 0)

# 創建多子圖佈局
fig = make_subplots(
    rows=num_rows, 
    cols=num_cols,
    subplot_titles=unique_songs,
    horizontal_spacing=0.05,  # 控制子圖之間的水平間距
    vertical_spacing=0.03      # 控制子圖之間的垂直間距
)

# 添加每首歌的主觀性柱狀圖到子圖中
for idx, song in enumerate(unique_songs):
    row = idx // num_cols + 1
    col = idx % num_cols + 1
    song_data = df[df['Song title'] == song]
    
    fig.add_trace(
        go.Bar(
            x=song_data['Stanza Number'],
            y=song_data['Subjectivity'],
            marker=dict(color=song_data['Subjectivity'], colorscale='Blues'),
            name=song
        ),
        row=row,
        col=col
    )

# 更新佈局設置，確保只有第一列顯示 y 軸標籤
fig.update_layout(
    height=200 * num_rows,
    title_text="Subjectivity of Stanzas in Each Song",
    showlegend=False
)

# 隱藏除第一列外的所有 y 軸標籤
for r in range(1, num_rows + 1):
    for c in range(1, num_cols + 1):
        if c != 1:  # 只有第一列保留 y 軸標籤
            fig.update_yaxes(title_text='', row=r, col=c)

            # 更新子圖標題的字體大小
for annotation in fig.layout.annotations:
    annotation.font.size = 12  # 調整子圖標題的字體大小
    
# 顯示圖表
fig.show()



In [ ]:

# 對每段歌詞進行情感分析，計算極性
df['Polarity'] = df['Stanza Text'].apply(lambda text: TextBlob(text).sentiment.polarity)

# 獲取所有唯一的歌曲名稱
unique_songs = df['Song title'].unique()

# 設置每行顯示的子圖數量，例如 4
num_cols = 4
num_rows = len(unique_songs) // num_cols + (1 if len(unique_songs) % num_cols > 0 else 0)

# 創建多子圖佈局
fig = make_subplots(
    rows=num_rows, 
    cols=num_cols,
    subplot_titles=unique_songs,
    horizontal_spacing=0.05,  # 控制子圖之間的水平間距
    vertical_spacing=0.03      # 控制子圖之間的垂直間距
)

# 添加每首歌的極性柱狀圖到子圖中
for idx, song in enumerate(unique_songs):
    row = idx // num_cols + 1
    col = idx % num_cols + 1
    song_data = df[df['Song title'] == song]
    
    fig.add_trace(
        go.Bar(
            x=song_data['Stanza Number'],
            y=song_data['Polarity'],
            marker=dict(color=song_data['Polarity'], colorscale='RdYlGn',),
            name=song
        ),
        row=row,
        col=col
    )

# 更新佈局設置，確保只有第一列顯示 y 軸標籤
fig.update_layout(
    height=200 * num_rows,
    title_text="Polarity of Stanzas in Each Song",
    title_font=dict(size=24),  # 調整圖表總標題的字體大小
    showlegend=False,
    margin=dict(l=50, r=50, t=100, b=50),
    font=dict(size=12)  # 調整整體字體大小，包含子圖標題、軸標籤等
)

# 為每一行的第一列設置 y 軸標籤 "Polarity"，其他列則隱藏
for r in range(1, num_rows + 1):
    for c in range(1, num_cols + 1):
        if c == 1:  # 只為第一列的子圖顯示 y 軸標籤
            fig.update_yaxes(title_text='Polarity', row=r, col=c)
        else:
            fig.update_yaxes(title_text='', row=r, col=c)

# 更新子圖標題的字體大小
for annotation in fig.layout.annotations:
    annotation.font.size = 12  # 調整子圖標題的字體大小

# 顯示圖表
fig.show()
